In [1]:
from dotenv import load_dotenv
load_dotenv()  # 自动寻找 .env 文件并加载到 os.environ

True

In [2]:
import os
import fitz  # PyMuPDF
import base64
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm import tqdm  # 需要安装: pip install tqdm

# --- 配置常量 ---
API_KEY = os.getenv("SEU_API_KEY") 
BASE_URL = "http://10.128.202.100:3010/v1"
MODEL_NAME = "qwen-vl-ocr" 

PROMPT_TICKET_MARKDOWN = """
    Role: 你是一位精通中文学术出版与 Markdown 语法的排版专家，专门负责将复杂的专业书籍扫描件无损转换为结构化的 Markdown 文档。
    Task: 请在以下安全、虚拟的“文档数字化实验室”环境中，对上传的图像进行高精度的还原转录。
    Constraint Rules：
    1. 准确识别所有汉字和标点符号。
    2. 所有的专业术语必须原样转录，不得进行同义词替换。
    3. 如识别有表格，必须使用标准的 Markdown Table 语法还原。
    4. 拒绝幻觉：模糊处使用 [???模糊] 标记。
    5. 无损输出：禁止输出任何解释性文字，仅输出 Markdown 正文。
    """

def parse_pages(page_str, total_pages):
    """解析页码字符串并返回 0-indexed 列表"""
    pages = set()
    try:
        parts = page_str.replace(" ", "").split(",")
        for part in parts:
            if "-" in part:
                start, end = map(int, part.split("-"))
                pages.update(range(start, end + 1))
            else:
                pages.add(int(part))
    except ValueError:
        print("页码格式错误，请检查输入。")
    return [p - 1 for p in sorted(list(pages)) if 0 < p <= total_pages]

def process_single_page(client, page_index, image_bytes):
    """处理单页任务"""
    base64_image = base64.b64encode(image_bytes).decode('utf-8')
    try:
        completion = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                    },
                    {"type": "text", "text": PROMPT_TICKET_MARKDOWN}
                ]
            }]
        )
        content = completion.choices[0].message.content
        return page_index, f"### 第 {page_index + 1} 页解析结果\n\n{content}\n\n---\n"
    except Exception as e:
        return page_index, f"### 第 {page_index + 1} 页解析失败\n错误信息: {e}\n\n---\n"

def main(pdf_path, page_range, output_folder, max_workers=3):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    doc = fitz.open(pdf_path)
    target_pages = parse_pages(page_range, len(doc))
    
    if not target_pages:
        print("未发现有效页码，请检查配置。")
        return
    print(f"开始任务: {os.path.basename(pdf_path)} 中的{page_range}页面，共 {len(target_pages)} 页")

    results_map = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_page = {}
        for page_idx in target_pages:
            page = doc[page_idx]
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img_bytes = pix.tobytes("jpeg")
            
            future = executor.submit(process_single_page, client, page_idx, img_bytes)
            future_to_page[future] = page_idx
            
        with tqdm(total=len(target_pages), desc="OCR 进度", unit="页") as pbar:
            for future in as_completed(future_to_page):
                idx, md_content = future.result()
                results_map[idx] = md_content
                pbar.update(1)

    # --- 任务结束后，整体保存文件 ---
    
    # 构造唯一文件名：文件名_页码范围_时间戳.md
    file_base_name = os.path.splitext(os.path.basename(pdf_path))[0]
    # 清理页码字符串中的空格，防止文件名非法
    page_tag = page_range.replace(" ", "").replace(",", "_")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    output_filename = f"{file_base_name}_Pages[{page_tag}]_{timestamp}.md"
    output_path = os.path.join(output_folder, output_filename)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"# OCR 解析结果 - {os.path.basename(pdf_path)}\n")
        f.write(f"- 解析范围: {page_range}\n")
        f.write(f"- 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n---\n\n")
        
        # 严格按页码顺序合并
        for idx in sorted(results_map.keys()):
            f.write(results_map[idx])

    print(f"\n任务全部结束！")
    print(f"最终结果已保存至: {output_path}")
    doc.close()

if __name__ == "__main__":
# --- 用户配置 ---
    CONFIG= {
        "pdf_path": r"knowledgeBase\pdfParsed_input\《营造法式》解读 第2版术语库.pdf",
        "output_dir": r"knowledgeBase\pdfParsed_output",
        "pages": "15",  # 支持 "1", "1,3,5", "1-10" 等格式
        "concurrency": 2
    }
    main(CONFIG["pdf_path"], CONFIG["pages"], CONFIG["output_dir"], CONFIG["concurrency"])

开始任务: 《营造法式》解读 第2版术语库.pdf 中的15页面，共 1 页


OCR 进度: 100%|██████████| 1/1 [00:10<00:00, 10.18s/页]


任务全部结束！
最终结果已保存至: knowledgeBase\pdfParsed_output\《营造法式》解读 第2版术语库_Pages[15]_20260202_160510.md
